# Cluster Marker Statistical Comparisons and Cell Type Counting in Mouse Bone Marrow Failure Model Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id`. You will learn how to load metadata, discover record sets and fields, extract and analyze data, visualize distributions, and summarize findings.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.89ma-b663/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.89ma-b663/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs. This step discovers the structure, record sets, fields, and columns using `mlcroissant` methods.

In [ ]:
# List all available record sets by @id
record_sets_info = dataset.record_sets
print("Available record sets and fields:")
record_sets_ids = []
for record_set in record_sets_info:
    print(f"- RecordSet @id: {record_set['@id']}   Name: {record_set.get('name', '')}")
    record_sets_ids.append(record_set['@id'])
    if 'fields' in record_set:
        print("  Fields:")
        for field in record_set['fields']:
            print(f"    Field @id: {field['@id']}   Name: {field.get('name', '')}   Type: {field.get('dataType', '')}")
            if 'columns' in field:
                print("      Columns:")
                for column in field['columns']:
                    print(f"        Column @id: {column['@id']}   Name: {column.get('name', '')}")


**Record Sample Example:**

Now, print a sample of records for each record set using its `@id`.

In [ ]:
for rs_id in record_sets_ids:
    print(f"\nSample records from RecordSet @id: {rs_id}")
    for i, x in enumerate(dataset.records(record_set=rs_id)):
        print(x)
        if i >= 2:  # Show only first 3 records per set
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing the record set and field `@id`s discovered above.

In [ ]:
# Extract each record set into a pandas DataFrame using @id
dataframes = {}

for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:  # Only add if data exists
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"\nColumns in RecordSet @id {rs_id}: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head())
    else:
        print(f"\nNo records loaded for RecordSet @id {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, and group data by attributes. Reference column/field `@id` from above.

In [ ]:
# Example: Choose the first record set with numeric columns
import numpy as np

selected_rs_id = None
numeric_field_id = None
group_field_id = None

for rs_id in dataframes:
    df = dataframes[rs_id]
    numeric_cols = [col for col in df.columns if df[col].dtype in [np.int64, np.float64] or df[col].apply(lambda x: isinstance(x,(int,float))).all()]
    if numeric_cols:
        selected_rs_id = rs_id
        numeric_field_id = numeric_cols[0]
        # Pick a likely group field (not numeric)
        non_numeric_cols = [col for col in df.columns if col != numeric_field_id]
        if non_numeric_cols:
            group_field_id = non_numeric_cols[0]
        break

print(f"Selected record set @id: {selected_rs_id}")
print(f"Numeric field @id: {numeric_field_id}")
print(f"Grouping field @id (example): {group_field_id}")

if selected_rs_id and numeric_field_id:
    df = dataframes[selected_rs_id]
    # Convert all values to numeric if possible
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    # Group by the group field and calculate means
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())


## 5. Visualization
Visualize data distributions or field relationships. See how numeric values and groupings are represented.

In [ ]:
# Visualization: Numeric field histogram and grouping bar chart
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field_id:
    df = dataframes[selected_rs_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id} in RecordSet {selected_rs_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id} in RecordSet {selected_rs_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook guided you through loading, overview, and exploratory analysis of the FAIR^2 dataset using `mlcroissant`. By referencing elements using their `@id`, we ensured precise and reproducible data processing. 

- You discovered available record sets and their structure.
- Extracted and analyzed numeric and grouped data from record sets.
- Visualized data distributions and relationships between fields.

For deeper analysis, consult the dataset documentation and explore more fields, columns, and record sets by their `@id`.